# 01 - Register Pushed / Downloaded AEMO ZIPs in Unity Catalog Control Log (Task 1)
Configured as **Task 1 (`download_to_landing`)** in your Multi-Task Databricks Workflow Job.
1. Initializes `workspace.bronze` schema, `/Volumes/workspace/bronze/landing` Volume, and `workspace.bronze.meta_file_control_log` table.
2. Checks `workspace.bronze.meta_file_control_log` for previously processed files.
3. **Checks `/Volumes/workspace/bronze/landing/_manual_upload/` FIRST** for `.zip` files pushed from Microsoft Fabric (`03_fabric_push_to_databricks`), moves them into `/Volumes/workspace/bronze/landing/Dispatch_SCADA/YYYY-MM-DD/`, and appends them to `workspace.bronze.meta_file_control_log` with `status = 'DOWNLOADED'`.
4. Also scans `/Volumes/workspace/bronze/landing/Dispatch_SCADA/` for any untracked `.zip` files and adds them to `workspace.bronze.meta_file_control_log`.

### Cell 1: Imports & Unity Catalog Configuration

In [0]:
import json
import re
import shutil
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests
from pyspark.sql import SparkSession
from pyspark.sql import types as T

spark = SparkSession.builder.getOrCreate()

# AEMO NEM operates strictly on AEST (UTC+10) year-round (no Daylight Saving)
NEM_TZ = timezone(timedelta(hours=10))

CATALOG = "aemo"
SCHEMA = "bronze"
VOLUME_NAME = "landing"
LANDING_BASE_DIR = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}")
MANUAL_UPLOAD_DIR = LANDING_BASE_DIR / "_manual_upload"
CONTROL_LOG_TABLE = f"{CATALOG}.{SCHEMA}.meta_file_control_log"

NEMWEB_FEEDS = {
    "Dispatch_SCADA": {
        "url": "https://nemweb.com.au/Reports/Current/Dispatch_SCADA/",
        "pattern": r"(PUBLIC_DISPATCHSCADA_[0-9_]+\.zip)",
    }
}

# Ensure Unity Catalog Schema, Managed Volume, and Control Table exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_LOG_TABLE} (
        file_name STRING,
        feed_name STRING,
        landing_date_folder STRING,
        landing_path STRING,
        file_size_bytes BIGINT,
        status STRING,
        downloaded_at_utc STRING,
        ingested_at_utc STRING,
        purged_at_utc STRING,
        bronze_row_count BIGINT
    )
    USING DELTA
""")
MANUAL_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Landing Volume (UC) : {LANDING_BASE_DIR}")
print(f"Fabric Push Folder  : {MANUAL_UPLOAD_DIR}")
print(f"Control Log Table   : {CONTROL_LOG_TABLE}")

### Cell 2: Check `workspace.bronze.meta_file_control_log` for Previously Processed Files

In [0]:
df_existing_log = spark.table(CONTROL_LOG_TABLE).toPandas()
known_files = set(df_existing_log["file_name"].dropna().unique())
print(f"Found {len(known_files):,} tracked files in {CONTROL_LOG_TABLE}.")
if not df_existing_log.empty:
    display(df_existing_log.sort_values("downloaded_at_utc", ascending=False).head(10))

### Cell 3: Stage Files Pushed from Fabric (`_manual_upload`) into Date-Partitioned Landing Folders
Checks `/Volumes/workspace/bronze/landing/_manual_upload/` first for `.zip` files pushed by `03_fabric_push_to_databricks`, moves them into `/Volumes/workspace/bronze/landing/Dispatch_SCADA/YYYY-MM-DD/`, and prepares their log rows for `workspace.bronze.meta_file_control_log`.

In [0]:
def extract_settlement_date_str(zip_filename: str, fallback_dt: datetime) -> str:
    """Extract the date from file name for partitioning"""
    match = re.search(r"_(\d{4})(\d{2})(\d{2})\d{4}_", zip_filename)
    if match:
        return f"{match.group(1)}-{match.group(2)}-{match.group(3)}"
    match_archive = re.search(r"#(\d{4})(\d{2})(\d{2})\d{4}\.zip", zip_filename)
    if match_archive:
        return f"{match_archive.group(1)}-{match_archive.group(2)}-{match_archive.group(3)}"
    return fallback_dt.strftime("%Y-%m-%d")

now_utc = datetime.now(timezone.utc)
now_nem = datetime.now(NEM_TZ)
new_log_records = []
skipped_count = 0
ingestion_mode = "NONE"

# 1. Check `_manual_upload` for files pushed from Microsoft Fabric
pushed_zips = sorted(MANUAL_UPLOAD_DIR.glob("*.zip"))
if pushed_zips:
    print(f"Found {len(pushed_zips)} ZIP file(s) in `{MANUAL_UPLOAD_DIR}` (pushed from Fabric).")
    ingestion_mode = "FABRIC_PUSH_TO_VOLUME"
    feed_name = "Dispatch_SCADA"

    for src_zip in pushed_zips:
        if src_zip.name in known_files:
            skipped_count += 1
            src_zip.unlink(missing_ok=True)
            continue

        date_folder = extract_settlement_date_str(src_zip.name, now_nem)
        target_dir = LANDING_BASE_DIR / feed_name / date_folder
        target_dir.mkdir(parents=True, exist_ok=True)
        target_path = target_dir / src_zip.name

        shutil.copy2(src_zip, target_path)
        fsize = target_path.stat().st_size
        src_zip.unlink(missing_ok=True)

        known_files.add(src_zip.name)
        print(f"  -> Staged pushed ZIP: {src_zip.name} ({fsize:,} bytes) -> {target_path}")

        new_log_records.append({
            "file_name": src_zip.name,
            "feed_name": feed_name,
            "landing_date_folder": date_folder,
            "landing_path": str(target_path),
            "file_size_bytes": int(fsize),
            "status": "DOWNLOADED",
            "downloaded_at_utc": now_utc.isoformat(),
            "ingested_at_utc": None,
            "purged_at_utc": None,
            "bronze_row_count": 0,
        })

# 2. Also scan `/Volumes/workspace/bronze/landing/Dispatch_SCADA/` for any untracked ZIPs
scada_landing_dir = LANDING_BASE_DIR / "Dispatch_SCADA"
if scada_landing_dir.exists():
    for existing_zip in sorted(scada_landing_dir.rglob("*.zip")):
        if existing_zip.name not in known_files:
            if ingestion_mode == "NONE":
                ingestion_mode = "EXISTING_VOLUME_FILES"
            date_folder = extract_settlement_date_str(existing_zip.name, now_nem)
            fsize = existing_zip.stat().st_size
            known_files.add(existing_zip.name)
            print(f"  -> Found untracked ZIP in Volume: {existing_zip.name} ({fsize:,} bytes)")

            new_log_records.append({
                "file_name": existing_zip.name,
                "feed_name": "Dispatch_SCADA",
                "landing_date_folder": date_folder,
                "landing_path": str(existing_zip),
                "file_size_bytes": int(fsize),
                "status": "DOWNLOADED",
                "downloaded_at_utc": now_utc.isoformat(),
                "ingested_at_utc": None,
                "purged_at_utc": None,
                "bronze_row_count": 0,
            })

print(f"\nNew files ready to log in {CONTROL_LOG_TABLE}: {len(new_log_records)} | Skipped (already logged): {skipped_count}")

### Cell 4: Append New Files to `workspace.bronze.meta_file_control_log` & Display Table

In [0]:
CONTROL_LOG_SCHEMA = T.StructType([
    T.StructField("file_name", T.StringType(), False),
    T.StructField("feed_name", T.StringType(), True),
    T.StructField("landing_date_folder", T.StringType(), True),
    T.StructField("landing_path", T.StringType(), True),
    T.StructField("file_size_bytes", T.LongType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("downloaded_at_utc", T.StringType(), True),
    T.StructField("ingested_at_utc", T.StringType(), True),
    T.StructField("purged_at_utc", T.StringType(), True),
    T.StructField("bronze_row_count", T.LongType(), True),
])

if new_log_records:
    df_new = pd.DataFrame(new_log_records)
    sdf_new = spark.createDataFrame(df_new, schema=CONTROL_LOG_SCHEMA)
    sdf_new.write.format("delta").mode("append").saveAsTable(CONTROL_LOG_TABLE)
    print(f"Appended {len(df_new)} new record(s) to {CONTROL_LOG_TABLE} (Mode: {ingestion_mode}).")
    display(df_new.head())
else:
    print("No new files were added to the control log in this run.")

result = {
    "status": "OK",
    "mode": ingestion_mode,
    "downloaded_files": len(new_log_records),
    "skipped_already_processed": skipped_count,
    "timestamp_utc": now_utc.isoformat(),
}

try:
    dbutils.jobs.taskValues.set(key="step_1_summary", value=result)
    dbutils.notebook.exit(json.dumps(result))
except Exception:
    pass